# TFL Generation Agent — Official 6-Study Library (DeepSeek)

**Uses the real, consolidated GitHub Library; no demo templates or fixture LLM responses.**

- **Target SAP:** a text-selectable PDF in your own Google Drive (not added to GitHub).
- **Library:** `library/` — **431 registered source templates from six studies** at the time this notebook was written (208 Tables, 20 Figures, 203 Listings).
- **Method:** Step 1 SAP parsing → Step 2 *live DeepSeek* evidence-based Table specification → Step 3 retrieval over **all approved official templates** → Step 4 safe generation attempt → Step 5 validation/export.
- **Current main-branch limitation:** Step 4 cannot yet adapt authentic bookmark-based DOCX files. The expected outcome is often `needs_review` **without** a generated shell. It never means retrieval failed, and it must not be reported as a completed DOCX.
- Only **Table** targets are supported by the current Step 2 / generator; Figure and Listing templates remain discoverable in the Library.

**Privacy:** Only transmit SAP excerpts to DeepSeek if authorized. Never commit confidential SAPs, API keys, outputs, or review notes to this public repository. Source Library files are read-only.

**Study-specific source QA / approvals are not equivalent to statistical correctness of a new target study.**

## 0 · Install the current Agent from GitHub `main`

A fresh Colab runtime is recommended. This notebook itself can live on a feature branch, while the Agent and the **latest official Library** are always cloned from `main`. The clone is isolated from previous demo folders.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
PROJECT_ROOT = Path("/content/tfl-generation-agent-official")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.tools.llm_client import DeepSeekJSONClient
from tfl_agent.tools.io_utils import write_json
import tfl_agent
import pandas as pd
from IPython.display import display, FileLink

assert Path(tfl_agent.__file__).resolve().is_relative_to(PROJECT_ROOT), (
    "An older tfl_agent module is cached. Restart the runtime and rerun from the first cell."
)
commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Agent loaded from:", tfl_agent.__file__)
print("GitHub main commit:", commit)
print("Project path:", PROJECT_ROOT)

## 1 · Audit the **official unified Library** (no API call)

This reads the consolidated root `library/`, **not** `library/studies/P-VCNA-003`, `examples/`, or `input/synthetic_sap.pdf`.

The existing Agent loader exposes the root through compatibility aliases (`metadata/Template_Metadata.json` and `templates/`). Those aliases point to the **official** unified Metadata and canonical DOCX collection. We also use `Template_Index.json` to attach the authoritative **per-study bookmark locator paths** in memory. Nothing is written into `library/`.

Only `review_status=approved` and `formal_use_eligible=true` records are passed to retrieval. Source approval does not grant automatic cross-study adaptation.

In [ ]:
import json
from collections import Counter

LIBRARY_PATH = PROJECT_ROOT / "library"  # OFFICIAL unified six-study Library
REGISTRY_PATH = LIBRARY_PATH / "Library_Registry.json"
TEMPLATE_INDEX_PATH = LIBRARY_PATH / "Template_Index.json"
CANONICAL_DOCX_ROOT = (LIBRARY_PATH / "TFL_Template_Library").resolve()

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
template_index = json.loads(TEMPLATE_INDEX_PATH.read_text(encoding="utf-8"))
index_records = {r["template_id"]: r for r in template_index["entries"]}
approved_ids = {
    r["template_id"] for r in template_index["entries"]
    if r.get("review_status") == "approved" and r.get("formal_use_eligible") is True
}

official_library = load_library(LIBRARY_PATH)
loader_format = official_library["format"]  # May show 'mvp' due to the compatibility alias.
assert registry["counts"]["registered_templates"] == len(official_library["templates"])
assert registry["counts"]["approved"] == len(approved_ids)
assert len(official_library["templates"]) >= 400, (
    "The official consolidated Library did not load: check GitHub main and symbolic links."
)

# Resolve canonical DOCX and per-study locators from the official Template_Index,
# without modifying Metadata, source files, or Agent code.
missing_locators = []
for template in official_library["templates"]:
    tid = template["identity"]["template_id"]
    entry = index_records[tid]
    expected_docx = (PROJECT_ROOT / entry["docx_path"]).resolve()
    actual_docx = Path(template["resolved_path"]).resolve()
    assert expected_docx == actual_docx, f"DOCX path mismatch: {tid}"
    assert actual_docx.is_file() and actual_docx.is_relative_to(CANONICAL_DOCX_ROOT), tid
    locator = (PROJECT_ROOT / entry["locator_map_path"]).resolve()
    if locator.is_file():
        template["locator_map_path"] = str(locator)  # In-memory path only.
    else:
        template["locator_map_path"] = None
        missing_locators.append(tid)

official_library["templates"] = [
    template for template in official_library["templates"]
    if template["identity"]["template_id"] in approved_ids
]
official_library["library_id"] = registry["library_id"]
official_library["format"] = "official_unified_v2_4_approved"
assert len(official_library["templates"]) == len(approved_ids)
assert len(official_library["templates"]) > 1

counts = Counter(t["identity"]["output_type"] for t in official_library["templates"])
study_counts = Counter(t["source"]["study_id"] for t in official_library["templates"])
keyword_mapping = official_library["index"].get("keyword_to_template_ids", {})
assert keyword_mapping, "Expected official global Keyword Index, not an empty demo index."
assert set(study_counts) == {s["study_id"] for s in registry["studies"]}

print("OFFICIAL LIBRARY ID:", official_library["library_id"])
print("Approved / registered:", len(approved_ids), "/", registry["counts"]["registered_templates"])
print("Original compatibility loader label:", loader_format, "(not the source data type)")
print("Global indexed keywords:", len(keyword_mapping))
print("Per-study locator maps missing:", len(missing_locators))
display(pd.DataFrame([{"Type": k, "Approved templates": v} for k, v in sorted(counts.items())]))
display(pd.DataFrame([{"Study": k, "Approved templates": v} for k, v in sorted(study_counts.items())]))
if missing_locators:
    print("First missing locator IDs:", missing_locators[:10])

## 2 · Load your real target SAP from Google Drive

The default path below matches the previous real-study experiment. You may change the path and physical PDF page range for a different study.

For `SAP_001_00.pdf`, the default `(4, 21)` selects **SAP body pages only**, excluding later Content List / source shell pages. Do **not** reuse this page range for a different PDF without checking it. Upload your PDF to Google Drive once; it is not sent to GitHub.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_SAP_RELATIVE_PATH = "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"
SAP_PATH = Path("/content/drive/MyDrive") / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)  # Physical PDF pages, 1-based; change for a different PDF.

if not SAP_PATH.is_file():
    folder = SAP_PATH.parent
    nearby = sorted(p.name for p in folder.glob("*.pdf")) if folder.exists() else []
    raise FileNotFoundError(
        f"Missing SAP at {SAP_PATH}. Folder exists: {folder.exists()}. "
        f"PDF names found: {nearby}. Change DRIVE_SAP_RELATIVE_PATH above."
    )
print("Real SAP:", SAP_PATH)
print("File size (MB):", round(SAP_PATH.stat().st_size / (1024 * 1024), 2))
print("Evidence page range:", SAP_BODY_PAGE_RANGE)

## 3 · Select target Table(s)

These are **target SAP TFL IDs / exact titles**, not source-Library template IDs. Test one Table first, then uncomment the second if needed. This MVP does not claim to enumerate every TFL in a study.

In [ ]:
TARGET_TABLES = [
    {"tfl_id": "14.1.3.1",
     "title": "Demographic and Baseline Characteristics – Safety Analysis Set"},
    # {"tfl_id": "14.2.3.1",
    #  "title": "Summary of Progression Free Survival – ITT Analysis Set"},
]
TOP_K = 10
MIN_SCORE = 0.55
OUTPUT_DIR = PROJECT_ROOT / "output" / "official_library_deepseek"
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"

assert 1 <= len(TARGET_TABLES) <= 2
for t in TARGET_TABLES:
    assert t["tfl_id"] and t["title"]
print("Targets:", TARGET_TABLES)
print("Retrieval scope:", len(official_library["templates"]), "approved official templates")
print("Output (never inside Library):", OUTPUT_DIR)

## 4 · Connect DeepSeek (real API — no fixture)

In Colab **Secrets** (key icon), set `DEEPSEEK_API_KEY` and grant notebook access. Never write the key directly into notebook source. DeepSeek calls may incur charges. Any failure stays visible; no synthetic fallback is used.

**Confirm you are authorized** to transmit excerpts of the selected SAP to this model provider.

In [ ]:
from google.colab import userdata
secret = userdata.get("DEEPSEEK_API_KEY")
if not secret:
    raise RuntimeError("Set DEEPSEEK_API_KEY in Colab Secrets and grant access to the notebook.")
llm = DeepSeekJSONClient(model="deepseek-flash", api_key=secret)
del secret
print("Live DeepSeek client initialized (key hidden).")

## 5 · Run the five-step pipeline against the **full official Library**

Step 1 loads the source SAP; Step 2 makes live DeepSeek calls and verifies cited SAP evidence. After Step 1 we attach the **audited, approved-only unified Library** in memory (instead of reusing the loader's raw compatibility aliases).

Step 3 searches all approved templates and records the best candidate, score, study of origin, and manual-review gates. A higher score after expanding to 431 templates must **not** be interpreted as a model probability or compared directly with the previous 0.8108 score.

The current `main` Step 4 supports synthetic `{{PLACEHOLDER}}` templates but not arbitrary source-bookmark adaptation. It is safe to call: real bookmarked sources should generally return `needs_review`, **without generating** a DOCX. Step 5 writes audit reports even in that case.

In [ ]:
import traceback

sessions = {}
rows = []
for target in TARGET_TABLES:
    tid = target["tfl_id"]
    safe_id = "".join(c if c.isalnum() or c in "._-" else "_" for c in tid)
    print("\n" + "=" * 80)
    print("TARGET:", tid, target["title"])
    session = TFLSession(
        sap_path=SAP_PATH,
        library_path=LIBRARY_PATH,
        schema_path=SCHEMA_PATH,
        prompts_dir=PROMPTS_DIR,
        output_dir=OUTPUT_DIR / safe_id,
        llm=llm,
        tfl_id=tid,
        title=target["title"],
        sap_body_page_range=SAP_BODY_PAGE_RANGE,
        allow_review=False,
    )
    sessions[tid] = session
    try:
        parsed = session.step1_parse()
        # Do NOT use a per-study or synthetic Library: this verified reference
        # contains all approved official templates and their per-study locators.
        session.library = official_library
        assert len(session.library["templates"]) == len(approved_ids)
        print("STEP 1 | SAP blocks:", parsed["block_count"])
        print("STEP 1 | Official approved Library count:", len(session.library["templates"]))

        spec = session.step2_build_specification()
        print("STEP 2 | Specification readiness:", spec["readiness"])
        print("STEP 2 | SAP evidence blocks:", len(spec.get("provenance", {}).get("evidence", [])))
        print("STEP 2 | Schema valid:", session.spec_validation["schema_valid"])
        print("STEP 2 | Completion audit:", (session.spec_debug or {}).get("completion", {}).get("attempted", False))
        display(pd.DataFrame([spec["statistical_requirements"]]).T.rename(columns={0:"Requirement"}))
        if spec.get("issues"):
            display(pd.DataFrame(spec["issues"]).head(12))

        result = session.step3_retrieve_templates(top_k=TOP_K, min_score=MIN_SCORE)
        print("STEP 3 | Retrieval status:", result["status"])
        print("STEP 3 | Best candidate:", result.get("best_candidate_template_id"))
        print("STEP 3 | Automatically eligible:", result.get("selected_template_id"))
        if result.get("reasons"):
            print("STEP 3 | Reasons:", "; ".join(result["reasons"]))
        candidates = result.get("candidates", [])
        if candidates:
            table = pd.DataFrame(candidates)
            selected_columns = [
                col for col in ("template_id", "score", "source_study_id",
                                "table_family", "title_similarity", "statistics_match",
                                "method_similarity", "population_change_required",
                                "requires_endpoint_review", "explanation")
                if col in table.columns
            ]
            display(table[selected_columns])
            best = candidates[0]
            source = index_records[best["template_id"]]
            print("Best-source study:", source["study_id"])
            print("Best-source canonical DOCX:", source["docx_path"])
            print("Best-source locator:", source["locator_map_path"])
            display(FileLink(str((PROJECT_ROOT / source["docx_path"]).resolve()),
                             result_html_prefix="View SOURCE template (not generated): "))
        else:
            print("No eligible output-family candidate found.")

        plan = session.preview_modification_plan()
        print("STEP 4 | Auto-adaptation safe:", plan.get("safe_to_generate", False))
        print("STEP 4 | Preview reason:", plan.get("reason", "No reason reported"))
        generation = session.step4_generate_shell()
        print("STEP 4 | Status:", generation["status"])
        if generation.get("issues"):
            print("STEP 4 | First issues:", generation["issues"][:5])

        report = session.step5_validate_and_export()
        print("STEP 5 | Status:", report["status"])
        if session.shell_path:
            display(FileLink(session.shell_path, result_html_prefix="Generated DRAFT DOCX: "))
        rows.append({
            "Target": tid, "Spec readiness": spec["readiness"],
            "Best candidate": result.get("best_candidate_template_id"),
            "Score": candidates[0]["score"] if candidates else None,
            "Source study": candidates[0].get("source_study_id") if candidates else None,
            "Retrieval status": result["status"], "Generation": report["status"],
            "DOCX generated": bool(session.shell_path),
        })
    except Exception as exc:
        # Never mask API failures, mistaken paths, or schema problems as successes.
        print("FAILED:", type(exc).__name__, str(exc))
        traceback.print_exc()
        rows.append({"Target": tid, "Spec readiness": None, "Best candidate": None,
                     "Score": None, "Source study": None, "Retrieval status": "failed",
                     "Generation": "failed", "DOCX generated": False})

print("\nFINAL RESULTS — OFFICIAL LIBRARY ONLY")
display(pd.DataFrame(rows))
assert all(r["Generation"] != "failed" for r in rows), (
    "At least one target failed. See traceback above before treating this as a completed test."
)

## 6 · Export a private audit ZIP

The generated report contains the target SAP evidence, selected templates, scores, rationale, and (if possible) a DOCX draft. It may contain confidential SAP excerpts. Save the ZIP privately. **Do not commit it to the public GitHub repository.** The files under `library/` are not changed.

In [ ]:
import shutil
from google.colab import files

assert OUTPUT_DIR.is_dir(), "Run Step 5 first."
archive = Path(shutil.make_archive(
    "/content/tfl_official_library_deepseek_audit", "zip", root_dir=OUTPUT_DIR
))
print("Audit ZIP:", archive)
print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))
files.download(str(archive))

## What a successful run proves — and what it does not

- **Valid Step 1–3 result:** the Agent used real target-SAP evidence and ranked **the official, approved, multi-study Library**. Retrieval uses deterministic heuristic scores, not calibrated probabilities.
- **Step 4 = needs_review:** a real bookmarked template was retrieved but could not be safely adapted by the `main` generator, or compatibility/specificity gates blocked use. Do **not** present the source DOCX as a generated target shell.
- **Clinical/statistical review remains necessary:** analysis population, denominator, endpoint, tables, headers, footnotes, layout, attribution, and correctness are not established by matching or basic schema checks.
- The earlier experimental bookmark adapter is on a separate **draft PR**, not merged into `main`. This Notebook intentionally uses current `main` with the consolidated official Library, rather than mixing code and Library from divergent GitHub branches.